In [4]:
import shutil
import random
from collections import defaultdict
import pandas as pd
# from google.colab import drive
import os

# 1) Mount Google Drive (akan minta otorisasi)
# drive.mount('/content/drive')


# ==============================
# KONFIGURASI PATH
# ==============================

# 2) Cek apakah folder sumber ada (ganti path jika berbeda)
SOURCE_DIR = r"D:\6S2\Thesis\7 dataset\hama1"   # Folder hasil Program 0
print("Exists?", os.path.exists(SOURCE_DIR))
if os.path.exists(SOURCE_DIR):
    print("Isi folder:", sorted(os.listdir(SOURCE_DIR))[:20])
else:
    print("Folder tidak ditemukan. Pastikan path ZIP diextract ke:", SOURCE_DIR)

TARGET_DIR = r"D:\6S2\Thesis\7 dataset\hama1_split"

SPLIT_RATIO = {
    "train": 0.70,
    "valid": 0.15,
    "test":  0.15
}

# Pastikan random stabil
random.seed(42)

# ==============================
# 1. BACA DATASET SUMBER
# ==============================

print("🔍 Membaca dataset dari:", SOURCE_DIR)

classes = sorted(os.listdir(SOURCE_DIR))

# cek apakah sudah benar struktur
print("\n=== Ditemukan class folder ===")
for c in classes:
    print(f"Class {c}")

# simpan semua file
dataset_map = {}

for cls in classes:
    cls_path = os.path.join(SOURCE_DIR, cls)
    images = sorted(os.listdir(cls_path))
    dataset_map[cls] = images

# tampilkan info awal
print("\n=== Informasi Jumlah Gambar ===")
info = []
total = 0

for cls, imgs in dataset_map.items():
    info.append([cls, len(imgs)])
    total += len(imgs)

df = pd.DataFrame(info, columns=["Class", "Jumlah"])
print(df)
print("\nTotal gambar:", total)

# ==============================
# 2. PREPARE FOLDER OUTPUT
# ==============================

print("\n📁 Mempersiapkan folder output...")

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

for subset in ["train", "valid", "test"]:
    os.makedirs(os.path.join(TARGET_DIR, subset), exist_ok=True)

# buat folder class di setiap subset
for subset in ["train", "valid", "test"]:
    for cls in classes:
        os.makedirs(os.path.join(TARGET_DIR, subset, cls), exist_ok=True)

print("Folder output siap.\n")

# ==============================
# 3. STRATIFIED SPLIT
# ==============================

print("✂️ Melakukan stratified split 70/15/15...\n")

split_stats = {
    "train": defaultdict(int),
    "valid": defaultdict(int),
    "test": defaultdict(int)
}

for cls in classes:
    files = dataset_map[cls]
    random.shuffle(files)

    n = len(files)
    n_train = int(n * SPLIT_RATIO["train"])
    n_valid = int(n * SPLIT_RATIO["valid"])
    n_test  = n - n_train - n_valid

    train_files = files[:n_train]
    valid_files = files[n_train:n_train+n_valid]
    test_files  = files[n_train+n_valid:]

    # ==============================
    # 4. COPY + RENAME FILES
    # ==============================

    # Train
    for idx, fname in enumerate(train_files):
        src = os.path.join(SOURCE_DIR, cls, fname)
        dst_name = f"{idx:04d}.jpg"
        dst = os.path.join(TARGET_DIR, "train", cls, dst_name)
        shutil.copy(src, dst)
        split_stats["train"][cls] += 1

    # Valid
    for idx, fname in enumerate(valid_files):
        src = os.path.join(SOURCE_DIR, cls, fname)
        dst_name = f"{idx:04d}.jpg"
        dst = os.path.join(TARGET_DIR, "valid", cls, dst_name)
        shutil.copy(src, dst)
        split_stats["valid"][cls] += 1

    # Test
    for idx, fname in enumerate(test_files):
        src = os.path.join(SOURCE_DIR, cls, fname)
        dst_name = f"{idx:04d}.jpg"
        dst = os.path.join(TARGET_DIR, "test", cls, dst_name)
        shutil.copy(src, dst)
        split_stats["test"][cls] += 1

print("✔️ Stratified split selesai.\n")

# ==============================
# 5. TAMPILKAN STATISTIK
# ==============================

print("=== STATISTIK HASIL SPLIT ===")

def print_stats(name):
    print(f"\n📌 {name.upper()} STATISTICS")
    rows = []
    total = 0
    for cls in classes:
        count = split_stats[name][cls]
        rows.append([cls, count])
        total += count
    df = pd.DataFrame(rows, columns=["Class", "Jumlah"])
    print(df)
    print("Total:", total)

print_stats("train")
print_stats("valid")
print_stats("test")

# print("\n🎉 Program 1 selesai. Dataset siap dipakai training!")


Exists? True
Isi folder: ['0', '1', '10', '11', '2', '3', '4', '5', '6', '7', '8', '9']
🔍 Membaca dataset dari: D:\6S2\Thesis\7 dataset\hama1

=== Ditemukan class folder ===
Class 0
Class 1
Class 10
Class 11
Class 2
Class 3
Class 4
Class 5
Class 6
Class 7
Class 8
Class 9

=== Informasi Jumlah Gambar ===
   Class  Jumlah
0      0     605
1      1     475
2     10     480
3     11     580
4      2     325
5      3     745
6      4     455
7      5     791
8      6     290
9      7    1110
10     8    1194
11     9     686

Total gambar: 7736

📁 Mempersiapkan folder output...
Folder output siap.

✂️ Melakukan stratified split 70/15/15...

✔️ Stratified split selesai.

=== STATISTIK HASIL SPLIT ===

📌 TRAIN STATISTICS
   Class  Jumlah
0      0     423
1      1     332
2     10     336
3     11     406
4      2     227
5      3     521
6      4     318
7      5     553
8      6     203
9      7     777
10     8     835
11     9     480
Total: 5411

📌 VALID STATISTICS
   Class  Jumlah
0     

In [7]:
import os
import zipfile
print("\n🎉 SPLIT SELESAI. Mempersiapkan ZIP...\n")

# ==============================
# 5. ZIP OUTPUT FOLDER
# ==============================
ZIP_OUTPUT = r"D:\6S2\Thesis\7 dataset\hama1_split.zip" 
if os.path.exists(ZIP_OUTPUT):
    os.remove(ZIP_OUTPUT)

print(f"📦 Membuat ZIP: {ZIP_OUTPUT}")

with zipfile.ZipFile(ZIP_OUTPUT, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(TARGET_DIR):
        for file in files:
            abs_path = os.path.join(root, file)
            rel_path = os.path.relpath(abs_path, os.path.dirname(TARGET_DIR))
            zipf.write(abs_path, rel_path)

print("✔️ ZIP selesai.")

# ==============================
# 6. HAPUS FOLDER SPLIT
# ==============================

print(f"🗑 Menghapus folder split: {TARGET_DIR}")
shutil.rmtree(TARGET_DIR)

print("\n🎉 Program 1 SELESAI dengan ZIP + Clean Up!")
print("File final:", ZIP_OUTPUT)


🎉 SPLIT SELESAI. Mempersiapkan ZIP...

📦 Membuat ZIP: D:\6S2\Thesis\7 dataset\hama1_split.zip
✔️ ZIP selesai.
🗑 Menghapus folder split: D:\6S2\Thesis\7 dataset\hama1_split

🎉 Program 1 SELESAI dengan ZIP + Clean Up!
File final: D:\6S2\Thesis\7 dataset\hama1_split.zip
